# Projet : Stage
### Objectif

Pour les attributs spatiaux et temporels, on voudrait faire la même chose. Pour cette partie, il y a 3 étapes à faire : 

1. Identifier les attributs spatiaux / temporels. 
2. Trouver le niveau d'hiérarchie de chaque attributs selon les hiérarchies spatiales / temporelles
3. Identifier le niveau d'hiérarchie le plus fin parmis tous les attributs spatiaux / temporels en tant que granularité minimum de dataset; identifier l'attribut au niveau d'hiérarchie le plus haut parmis tous les attributs en tant que scope de dataset et donner la liste de ses valeurs distinctes. 

L'output final qu'on demande est un dossier json de métadonnée de tous les datasets.

## 1. Hiérarchisation des données

In [ ]:
import os
import re
import json
import csv
import pandas as pd
import load_file as lf
import uml_class as uml

def construire_dictionnaire_hierarchise():

    def fillIris(dic_hierarchise):
        iris_df = pd.read_csv('table_passage_1999_2022.csv', sep=',', encoding='utf-8')
        iris_values = iris_df.dropna().values.astype(str).tolist()
        listeIris = []
        for iris in iris_values:
            for elt in iris:
                if elt not in listeIris and elt != "":
                    listeIris.append(elt.lower())
        
        dic_hierarchise['iris'] = listeIris
        return
    
    def fillCodePostaux(dic_hierarchise):
        code_postaux = pd.read_csv("019HexaSmal.csv", sep=';', encoding='latin-1')
        code_values = code_postaux["Code_postal"]
        code_insee = code_postaux["#Code_commune_INSEE"]
        cleaned_codes = code_values.dropna().astype(str).str.strip().unique().tolist()
        cleaned_insee = code_insee.dropna().astype(str).str.strip().unique().tolist()
        dic_hierarchise["communes"].append(sorted(cleaned_codes))
        dic_hierarchise["communes"].append(sorted(cleaned_insee))

        return

    def fillEPCI(dic_hierarchise):
        df = pd.read_excel("epci-excel.xlsx", dtype=str)
    
        code_epci = df["CODE_EPCI"].dropna().astype(str).str.strip()
        nom_epci = df["NOM_EPCI"].dropna().astype(str).str.strip()

        all_epci = set(code_epci.tolist() + nom_epci.tolist())

        existing_epci = set(dic_hierarchise.get("epci", []))
        updated_epci = sorted(existing_epci.union(all_epci))

        dic_hierarchise["epci"] = updated_epci

        return

    def fillChamp(dicChamps, dic_hierarchise, rang):
        for i in range(len(dicChamps['features'])):
            champ = dicChamps['features'][i]['properties']['nom']
            if champ not in dic_hierarchise[rang]:
                dic_hierarchise[rang].append(champ.lower())
        return

    def fillDictionnaireGeoJSON():
        fichiers = ['communes', 'departements', 'regions']
        dic_hierarchise = {}

        for fichier in fichiers:
            dic_hierarchise[fichier] = []
            with open(f"levels/france-geojson/{fichier}-avec-outre-mer.geojson", "r", encoding="utf-8") as mon_json:
                data = json.load(mon_json)
                fillChamp(data, dic_hierarchise, fichier)
        
        dic_hierarchise['regions'].append("ile-de-france")

        return dic_hierarchise

    def fillDictionnaireQuartiers(dic_hierarchise):
        with open("liste-correspondance-qp2024-qp2015.csv", "r", encoding="utf-8") as fichier:
            reader = csv.reader(fichier, delimiter=";")
            listeQuartiers = list(reader)[1:]  # Ignorer l'en-tête

        dic_hierarchise['quartiers'] = []
        for i in range(len(listeQuartiers)):
            quartier = listeQuartiers[i][1]
            if quartier not in dic_hierarchise['quartiers'] and quartier != "":
                dic_hierarchise['quartiers'].append(quartier.lower())
                
        for i in range(len(listeQuartiers)):
            quartier = listeQuartiers[i][3]
            if quartier not in dic_hierarchise['quartiers'] and quartier != "":
                dic_hierarchise['quartiers'].append(quartier.lower())

    dic_hierarchise = fillDictionnaireGeoJSON()
    fillDictionnaireQuartiers(dic_hierarchise)
    fillIris(dic_hierarchise)
    fillEPCI(dic_hierarchise)
    fillCodePostaux(dic_hierarchise)

    champs = ['regions', 'departements', 'communes', 'quartiers', 'QP', 'iris', 'epci']
    dic_hierarchise = {champ: dic_hierarchise[champ] for champ in champs if champ in dic_hierarchise}
    dic_hierarchise['pays'] = ['france', 'france métropolitaine', 'france d\'outre-mer', 'france entière']
    
    return dic_hierarchise

def recuperer_dictionnaire_hierarchise():
    try:
        with open("dic_hierarchise.json", "r", encoding="utf-8") as fichier:
            dic_hierarchise = json.load(fichier)
    except FileNotFoundError:
        dic_hierarchise = construire_dictionnaire_hierarchise()
        with open("dic_hierarchise.json", "w", encoding="utf-8") as fichier:
            json.dump(dic_hierarchise, fichier, ensure_ascii=False, indent=4)
    
    return dic_hierarchise

In [ ]:
dic_hierarchise = recuperer_dictionnaire_hierarchise()

champs_ranges = ['pays', 'regions', 'departements', 'epci', 'quartiers', 'communes', 'iris', 'geopoints']
temps_ranges = ['annee', 'trimestre', 'mois', 'semaine', 'date']

hierarchie_champs_spa = {champ: (len(champs_ranges) - i) for i, champ in enumerate(champs_ranges)}
hierarchie_temps_spa = {temps: (len(temps_ranges) - i) for i, temps in enumerate(temps_ranges)}

## 2. Identification des attributs spatiaux dans un fichier csv/xlsx

### 2.1 Récupérer tous les attributs spatiaux

#### 2.1.1 Recuperation de tous les datasets

In [ ]:
import os

def getFiles(origine='Opendata'):
    fichiers = []

    for dossier in os.walk(origine):
        for fichier in dossier[2]:
            if fichier.endswith('.csv') or fichier.endswith('.xlsx'):
                fichiers.append(os.path.join(dossier[0], fichier))
    
    return sorted(fichiers)

datasets = getFiles()

#### 2.1.2 Recherche des attributs spatiaux via contenu des cellules

Variables utiles

In [ ]:
regexAnnee = r'(19\d{2}|20\d{2})$'
regexMois = r'^(0[1-9]|1[0-2])$'
regexJour = r'(0[1-9]|[12]\d|3[01])'
regexDate = r'(' + regexAnnee[:-1] + r'[-/]?' + regexMois[1:-1] + r'[-/]?' + regexJour + r')|(' + regexJour + r'[-/]' + regexMois[1:-1] + r'[-/]' + regexAnnee[:-1] + r')|('+ regexMois[1:-1] + r'[-/]' + regexAnnee[:-1] + r')'
regexHeure = r'(([10]\d)|(2[0-3]))[:h]([0-5]\d)([:h]([0-5]\d))?'
regexTrim = r'(19\d{2}|20\d{2})_[a-zA-Z]{1}[1-3]'

listeRegexTemporel = [[regexDate, 'date'], [regexAnnee, 'annee'], [regexTrim, 'trimestre']]

Fonctions utiles

In [ ]:
import pandas as pd
import re

def estGeopoint(cell):
    cell = str(cell).strip()
    match = re.match(r'^(-?\d+(?:\.\d+)?)[,; ]\s*(-?\d+(?:\.\d+)?)$', cell)
    
    if match:
        try:
            lat, lon = float(match.group(1)), float(match.group(2))
            if -90 <= lat <= 90 and -180 <= lon <= 180:
                return [True, 'geopoints']
        except Exception:
            pass
    return [False, None]

def estSpatial(cell):
    cell = str(cell).lower()
    infoGeopoint = estGeopoint(cell)
    if infoGeopoint[0]:
        return infoGeopoint
    
    for champ, valeurs in dic_hierarchise.items():
        if cell in valeurs:
            return [True, champ]
        
    return [False, None]

def estTemporel(cell):
    if not isinstance(cell, str):
        cell = str(cell)
    for regex, label in listeRegexTemporel:
        if re.match(regex, cell):
            return [True, label]
    if cell.lower() in ['janvier', 'fevrier', 'mars', 'avril', 'mai', 'juin', 'juillet', 'aout', 'septembre', 'octobre', 'novembre', 'decembre']:
        return [True, 'mois']
    return [False, None]

def recupererAttributsSpatiaux(headers, df, score):
    n_rows = min(100, df.shape[0]) 
    liste_attributs_spatiaux = {}

    # Rechercher en fonction des headers
    for i, header in enumerate(headers):
        accents = {
            r'[àáâãäå]': 'a',
            r'[èéêë]': 'e',
            r'[ìíîï]': 'i',
            r'[òóôõö]': 'o',
            r'[ùúûü]': 'u',
            r'[ç]': 'c',
            r'[œ]': 'oe',
            r'[æ]': 'ae'
        }
        header_lower = header.lower()
        for pattern, repl in accents.items():
            header_lower = re.sub(pattern, repl, header_lower)
        # Recherche plus stricte pour lat/lon : doit être un mot entier, pas une sous-chaîne
        if any(x in header_lower for x in ['geopoint', 'geopoints', 'coordonnees', 'latitude-longitude', 'lat-long', 'latlon', 'lattitude', 'longitude', 'adresse', 'adresse geographique', 'adresse geographique complete']):
            liste_attributs_spatiaux[header] = ["header", 'geopoints']
        # Pour "lat" et "lon", on exige que ce soit un mot entier (évite "longueur", "lateral", etc.)
        elif re.fullmatch(r'(lat|latitude)', header_lower):
            liste_attributs_spatiaux[header] = ["header", 'geopoints']
        elif re.fullmatch(r'(lon|lng|longitude)', header_lower):
            liste_attributs_spatiaux[header] = ["header", 'geopoints']
        elif any(x in header_lower for x in ['iris', 'code iris', 'iris code', 'iris code insee']):
            liste_attributs_spatiaux[header] = ["header", 'iris']
        elif any(x in header_lower for x in ['code postal', 'code postaux', 'postal', 'code postal insee', 'insee code postal', 'commune']):
            liste_attributs_spatiaux[header] = ["header", 'communes']
        elif any(x in header_lower for x in ['epci', 'code epci', 'epci code', 'epci code insee']):
            liste_attributs_spatiaux[header] = ["header", 'epci']
        elif any(x in header_lower for x in ['quartier prioritaire', 'quartiers prioritaires', 'code qp', 'qp code', 'quartier', 'quartiers', 'code quartier', 'quartier code', 'quartier code insee']):
            liste_attributs_spatiaux[header] = ["header", 'quartiers']
        elif any(x in header_lower for x in ['region', 'regions', 'code region', 'region code', 'region code insee']):
            liste_attributs_spatiaux[header] = ["header", 'regions']
        elif any(x in header_lower for x in ['departement', 'departements', 'code departement', 'departement code', 'departement code insee']):
            liste_attributs_spatiaux[header] = ["header", 'departements']
        elif any(x in header_lower for x in ['pays', 'pays de résidence', 'pays de naissance', 'pays d\'origine']):
            liste_attributs_spatiaux[header] = ["header", 'pays']        

    # Rechercher dans les valeurs des colonnes
    for j, header in enumerate(headers):
        col_values = df[header].astype(str).str.lower().head(n_rows)
        for cell in col_values:
            info = estSpatial(cell)
            if info[0]:
                score[j] += 1
                if score[j] * 10 >= 50 and header not in liste_attributs_spatiaux:
                    liste_attributs_spatiaux[header] = [cell, info[1]]
                    break
            elif header.lower() == 'iris' and header not in liste_attributs_spatiaux:
                liste_attributs_spatiaux[header] = [cell, 'iris']

    return liste_attributs_spatiaux

def recupererAttributsTemporels(headers, df, score):

    n_rows = min(100, df.shape[0]) 
    liste_attributs_temporels = {}

    # Rechercher en fonction des headers
    for i, header in enumerate(headers):
        if any(x in header.lower() for x in ['annee', 'annees', 'year', 'years', 'année', 'années']):
            liste_attributs_temporels[header] = ["header", 'annee']
        elif any(x in header.lower() for x in ['mois', 'month', 'months']):
            liste_attributs_temporels[header] = ["header", 'mois']
        elif any(x in header.lower() for x in ['trimestre', 'trimester', 'trimesters', 'semestre', 'semester']):
            liste_attributs_temporels[header] = ["header", 'trimestre']
        elif any(x in header.lower() for x in ['date', 'datetime', 'timestamp']):
            liste_attributs_temporels[header] = ["header", 'date']
        elif any(x in header.lower() for x in ['semaine', 'week', 'weeks']):
            liste_attributs_temporels[header] = ["header", 'semaine']
    
    # Rechercher dans les valeurs des colonnes
    for j, header in enumerate(headers):
        col_values = df[header].astype(str).head(n_rows)
        for cell in col_values:
            info = estTemporel(cell)
            if info[0]:
                if info[1] == 'annee':
                    try:
                        vals = df[header].astype(str)
                        if not vals.apply(lambda x: x.isdigit() and 1900 <= int(x) <= 2100).all():
                            break
                    except Exception:
                        break
                score[j] += 1
                if score[j] * 10 >= 50 and header not in liste_attributs_temporels:
                    liste_attributs_temporels[header] = [cell, info[1]]
                    break
    return liste_attributs_temporels

def recupererAttributsQualitatifs(headers, df, liste_attributs_spatiaux, liste_attributs_temporels):
    """
    Identifie les colonnes contenant des attributs qualitatifs (catégoriels),
    incluant les index numériques et les jours du mois.
    
    Args:
        headers: Liste des en-têtes de colonnes
        df: DataFrame pandas à analyser
        liste_attributs_spatiaux: Dictionnaire des attributs spatiaux identifiés
        liste_attributs_temporels: Dictionnaire des attributs temporels identifiés
        
    Returns:
        Une liste des attributs qualitatifs identifiés
    """
    n_rows = min(100, df.shape[0])
    liste_attributs_qualitatifs = []
    
    for header in headers:
        # Ignorer les colonnes déjà identifiées comme spatiales ou temporelles
        if header in liste_attributs_spatiaux or header in liste_attributs_temporels:
            continue
            
        # Récupérer les valeurs de la colonne
        col_values = df[header].head(n_rows)
        
        # Convertir en string, retirer les valeurs 'nan' et valeurs vides
        str_values = col_values.astype(str)
        clean_values = str_values[~str_values.isin(['', 'nan', 'NaN', 'None', 'NONE', 'null', 'Null', 'NULL'])]
        
        if len(clean_values) == 0:
            continue
            
        # Nombre de valeurs uniques
        unique_values = clean_values.nunique()
        ratio_unique = unique_values / len(clean_values) if len(clean_values) > 0 else 0
        
        # CAS 1: Texte avec peu de valeurs uniques - clairement catégoriel
        if 0 < ratio_unique < 0.5:
            try:
                # Vérifier si les valeurs sont numériques
                numeric_values = pd.to_numeric(clean_values)
                
                # Vérifier si c'est un jour du mois (1-31)
                if numeric_values.min() >= 1 and numeric_values.max() <= 31:
                    liste_attributs_qualitatifs.append(header)
                    continue
                    
                # Si les valeurs sont numériques mais en petit nombre (< 20 valeurs distinctes)
                # c'est probablement une variable catégorielle
                if unique_values < 20:
                    liste_attributs_qualitatifs.append(header)
            except:
                # Si la conversion échoue, c'est probablement du texte catégoriel
                liste_attributs_qualitatifs.append(header)
                
        # CAS 2: Texte court mais pas trop de valeurs uniques
        elif col_values.astype(str).str.len().mean() < 30 and unique_values < 50:
            liste_attributs_qualitatifs.append(header)
            
        # CAS 3: Séquences d'entiers potentiellement ID/index
        else:
            try:
                numeric_values = pd.to_numeric(clean_values)
                
                # Vérifier si les valeurs sont principalement des entiers
                if (numeric_values % 1 == 0).mean() > 0.95:
                    sorted_values = sorted(numeric_values.dropna().unique())
                    
                    if len(sorted_values) > 1:
                        # Calculer les écarts
                        gaps = [sorted_values[i+1] - sorted_values[i] for i in range(len(sorted_values)-1)]
                        avg_gap = sum(gaps) / len(gaps)
                        
                        # Un index a souvent des écarts très proches de 1 ou constants
                        if 0.8 < avg_gap < 1.2 or (max(gaps) - min(gaps) < 2):
                            liste_attributs_qualitatifs.append(header)
                            continue
                            
                        # Autre critère: si le nom contient des indices d'ID
                        if any(x in header.lower() for x in ['id', 'code', 'key', 'index', 'numero', 'num', 'identifiant']):
                            liste_attributs_qualitatifs.append(header)
            except:
                pass
                
    return liste_attributs_qualitatifs

def recupererAttributsQuantitatifs(headers, df, liste_attributs_spatiaux, liste_attributs_temporels, liste_attributs_qualitatifs):
    """
    Identifie les colonnes contenant des attributs quantitatifs (numériques continus).
    Distingue entre variables quantitatives et identifiants/index numériques.
    
    Args:
        headers: Liste des en-têtes de colonnes
        df: DataFrame pandas à analyser
        liste_attributs_spatiaux: Dictionnaire des attributs spatiaux déjà identifiés
        liste_attributs_temporels: Dictionnaire des attributs temporels déjà identifiés
        liste_attributs_qualitatifs: Liste des attributs qualitatifs déjà identifiés
        
    Returns:
        Une liste des attributs quantitatifs identifiés
    """
    n_rows = min(100, len(df))  # Analyse un échantillon représentatif
    liste_attributs_quantitatifs = []
    
    for header in headers:
        # Ignorer les colonnes déjà identifiées comme spatiales, temporelles ou qualitatives
        if (header in liste_attributs_spatiaux or 
            header in liste_attributs_temporels or 
            header in liste_attributs_qualitatifs):
            continue
            
        # Récupérer les valeurs de la colonne
        col_values = df[header].head(n_rows)
        
        # Essayer de convertir en numérique
        try:
            numeric_values = pd.to_numeric(col_values)
            
            # Ignorer les colonnes avec trop de valeurs manquantes
            if numeric_values.isna().mean() > 0.5:
                continue
                
            # Vérifier s'il s'agit de données numériques continues
            unique_values = numeric_values.nunique()
            non_na_count = len(numeric_values.dropna())
            ratio_unique = unique_values / non_na_count if non_na_count > 0 else 0
            
            # Détection d'identifiants ou index numériques
            # 1. Vérifier si les valeurs sont majoritairement entières
            is_mostly_integer = (numeric_values % 1 == 0).mean() > 0.95
            
            # 2. Vérifier la distribution des écarts entre valeurs consécutives
            if is_mostly_integer and unique_values >= 5:
                sorted_values = sorted(numeric_values.dropna().unique().tolist())
                gaps = [sorted_values[i+1] - sorted_values[i] for i in range(len(sorted_values)-1)]
                
                # Si les écarts sont variables mais pas trop grands
                if max(gaps) / (sum(gaps)/len(gaps) if gaps else 1) < 10:
                    # Si ce sont des nombres entiers avec des écarts irréguliers mais relativement petits
                    # C'est probablement une variable quantitative ordinale
                    liste_attributs_quantitatifs.append(header)
                # Si trop d'écarts très grands par rapport à la moyenne, c'est probablement un identifiant
                else:
                    continue
                    
            # Pour les nombres à virgule ou avec beaucoup de valeurs uniques -> quantitatif
            elif (not is_mostly_integer) or ratio_unique > 0.5 or unique_values > 20:
                liste_attributs_quantitatifs.append(header)
            
        except:
            # Si la conversion échoue, ce n'est pas une variable numérique
            continue
    
    return liste_attributs_quantitatifs

def rechercherLowGranEtScope(liste_attributs, hierarchie):
    if not liste_attributs:
        return {'LowGranularite': [None, None], 'Scope': [None, None]}
    
    min_att = list(hierarchie.keys())[0]
    max_att = list(hierarchie.keys())[-1]
    le_plus_bas = [min_att, None]
    le_plus_haut = [max_att, None]

    for champ, valeur in liste_attributs.items():
        if hierarchie[valeur[1]] <= hierarchie[le_plus_bas[0]]:
            le_plus_bas = [valeur[1], champ]
        if hierarchie[valeur[1]] >= hierarchie[le_plus_haut[0]]:
            le_plus_haut = [valeur[1], champ]

    result = {'LowGranularite': le_plus_bas, 'Scope': le_plus_haut}
    
    return result

def chercherEntete(df, max_lignes=20):
    lignes_testees = 0
    old_df = None
    while lignes_testees < max_lignes:
        headers = df.columns.tolist()
        headers_valides = all(
            not re.match(r'(?i:unnamed|nan)', str(h)) and ' ' not in str(h).strip()
            for h in headers
        )
        if headers_valides:
            if any((estTemporel(header)[0] or estSpatial(header)[0]) and old_df is not None for header in headers):
                df = old_df.copy()
            return df, False
        if len(df) < 1:
            break
        new_headers = df.iloc[0].tolist()
        old_df = df.copy()
        df = df[1:].copy()
        df.columns = [str(h) for h in new_headers]
        lignes_testees += 1
    return df, True

def spatialScopeToDict(scope, dataset, headers):
    if scope[1] is None:
        return {'spatialScopeLevel': None, 'spatialScope': None}
    
    else :
        scope_level = scope[0]
        scope_values = list(dataset[scope[1]].astype(str).unique())

    return {
        'spatialScopeLevel': scope_level, 
        'spatialScope': scope_values
    }

def temporalScopeToDict(scope, dataset, headers):
    if scope[1] is None:
        return {'temporalScopeLevel': None, 'temporalScopeStart': None, 'temporalScopeEnd': None}
    
    elif scope[1] == 'entêtes':
        scope_level = scope[0]
        scope_values = [col for col in headers if estTemporel(col)[0]]
        
    else:
        scope_level = scope[0]
        match scope[0]:
            case 'annee':
                scope_values = sorted([value for value in list(dataset[scope[1]].astype(str).unique()) if re.match(regexAnnee, value)])
            case 'trimestre':
                scope_values = sorted([value for value in list(dataset[scope[1]].astype(str).unique()) if re.match(regexTrim, value)])
            case 'mois':
                listeMois = ['janvier', 'fevrier', 'mars', 'avril', 'mai', 'juin', 'juillet', 'aout', 'septembre', 'octobre', 'novembre', 'decembre']
                scope_values = sorted([value.lower() for value in list(dataset[scope[1]].astype(str).unique()) if value.lower() in listeMois], key=listeMois.index)
            case 'date':
                scope_values = sorted([value for value in list(dataset[scope[1]].astype(str).unique()) if re.match(regexDate, value)])
    
    return {
        'temporalScopeLevel': scope_level,
        'temporalScopeStart': scope_values[0] if scope_values else None,
        'temporalScopeEnd': scope_values[-1] if scope_values else None
    }

def creerDatasetUML(dataset, nom_fichier, extension, granAndScopeSpat, granAndScopeTemp, liste_attributs_spatiaux, liste_attributs_temporels, liste_attributs_qualitatifs, liste_attributs_quantitatifs, headers, dic_headers={}, dic_informations={}):
    monSpatialScope = uml.DS_Spatial_Scope(None, None).from_dict(spatialScopeToDict(granAndScopeSpat['Scope'], dataset, headers))
    monTemporalScope = uml.DS_Temporal_Scope(None, None, None).from_dict(temporalScopeToDict(granAndScopeTemp['Scope'], dataset, headers))

    if len(dic_informations.keys()) > 0:
        title = dic_informations["title"]
        description = dic_informations["description"]
        sourceName = dic_informations["sourceName"]
        sourceType = dic_informations["sourceType"]

    data_content = []
    for champ, valeur in liste_attributs_spatiaux.items():
        dataName = champ
        dataDescription = dic_headers[champ] if champ in dic_headers.keys() else None
        dataType = "Spatial"
        parameterCode = "SPAT_" + champ.replace(" ", "_").upper()
        spatialLevel = valeur[1]
        spatialParam = uml.Spatial_Parameter(dataName, dataDescription, dataType, parameterCode, spatialLevel)
        data_content.append(spatialParam)
    
    for champ, valeur in liste_attributs_temporels.items():
        dataName = champ
        dataDescription = dic_headers[champ] if champ in dic_headers.keys() else None
        dataType = "Temporal"
        parameterCode = "TEMP_" + champ.replace(" ", "_").upper()
        temporalLevel = valeur[1]
        temporalParam = uml.Temporal_Parameter(dataName, dataDescription, dataType, parameterCode, temporalLevel)
        data_content.append(temporalParam)

    for attr in liste_attributs_qualitatifs:
        dataName = attr
        dataDescription = dic_headers[attr] if attr in dic_headers.keys() else None
        dataType = "Qualitative"
        parameterCode = "QUAL_" + attr.replace(" ", "_").upper()
        qualitativeParam = uml.Existing_Indicator(dataName, dataDescription, dataType, parameterCode, uml.Theme(None, None))
        data_content.append(qualitativeParam)

    for attr in liste_attributs_quantitatifs:
        dataName = attr
        dataDescription = dic_headers[attr] if attr in dic_headers.keys() else None
        dataType = "Quantitative"
        parameterCode = "QUAN_" + attr.replace(" ", "_").upper()
        quantitativeParam = uml.Existing_Indicator(dataName, dataDescription, dataType, parameterCode, uml.Theme(None, None))
        data_content.append(quantitativeParam)

    monDataset = uml.Dataset(
        title, description, None, extension, None, sourceName, sourceType, None,
        granAndScopeSpat['LowGranularite'][0], monSpatialScope,
        granAndScopeTemp['LowGranularite'][0], monTemporalScope,
        uml.Theme(None, None), data_content
    )
    
    monDataset.save_to_json(f'metadatas/{nom_fichier}.json')
    return

def chercherDataDescriptionXLSX(dataset, sheet, headers, listeSheets):
    patternDescription = r"(?i)(description\s+(des\s+)?(données|variables)|variable|liste\s+des\s+variables)"
    dic_headers = {}

    desc_sheet = next((s for s in listeSheets if re.match(patternDescription, s)), None)
    desc_format = 'sheet' if desc_sheet else 'header'
    sheet_to_read = desc_sheet if desc_sheet else sheet

    df = pd.read_excel(dataset, sheet_name=sheet_to_read, nrows=len(headers)+10, engine="openpyxl", header=None).dropna(how='all')

    ligneDesc = None
    for i in range(len(df)):
        row = df.iloc[i].astype(str).tolist()
        if row == headers:
            ligneDesc = i - 1 
            break

    if desc_format == 'sheet':
        for index, row in df.iterrows():
            row_lower = row.astype(str).str.lower().tolist()
            for header in headers:
                try:
                    idx = row_lower.index(header.lower())
                    description = row.iloc[idx + 1] if idx + 1 < len(row) else ""
                    dic_headers[header] = str(description).replace('\n', ' ').strip()
                except ValueError:
                    continue
                
    else:        
        if ligneDesc is not None and ligneDesc >= 0:
            listeDescriptions = df.iloc[ligneDesc].astype(str).tolist()
            if "nan" not in listeDescriptions:
                for i, header in enumerate(headers):
                    if i < len(listeDescriptions):
                        dic_headers[header] = listeDescriptions[i].replace('\n', ' ').strip()
                    else:
                        dic_headers[header] = ""
        else:
            return dic_headers, ligneDesc
    return dic_headers, ligneDesc

def process_dataframe(df, nom_fichier, extension, hierarchie_temps_spa, hierarchie_champs_spa, dataset=None):
    headers = df.columns.tolist()
    dic_informations = {}
    score_colonne = {k: 0 for k in range(len(headers))}
    df_sample = df.head(100).copy()
    df_sample = df_sample.astype(str)
    liste_attributs_spatiaux = recupererAttributsSpatiaux(headers, df_sample, score_colonne)
    liste_attributs_temporels = recupererAttributsTemporels(headers, df_sample, score_colonne)
    liste_attributs_qualitatifs = recupererAttributsQualitatifs(headers, df_sample, liste_attributs_spatiaux, liste_attributs_temporels)
    liste_attributs_quantitatifs = recupererAttributsQuantitatifs(headers, df_sample, liste_attributs_spatiaux, liste_attributs_temporels, liste_attributs_qualitatifs)

    for header in headers:
        if estTemporel(header)[0] and "entêtes" not in liste_attributs_temporels:
            liste_attributs_temporels["entêtes"] = [header, estTemporel(header)[1]]
    
    if "sheet" in nom_fichier:
        indexSheet = int(nom_fichier.split("sheet")[-1])
        dic_headers, ligneDesc = chercherDataDescriptionXLSX(dataset, indexSheet, headers, pd.ExcelFile(dataset, engine="openpyxl").sheet_names)

        if ligneDesc is not None and ligneDesc >= 0:
            dataframe_informations = pd.read_excel(dataset, sheet_name=indexSheet, nrows=ligneDesc, engine="openpyxl", header=None)
            
            dic_informations["title"] = dataframe_informations.iloc[0, 0] if len(dataframe_informations) > 0 else nom_fichier
            dic_informations["description"] = dataframe_informations.iloc[1, 0] if len(dataframe_informations) > 1 else None
            dic_informations["sourceName"] = "INCONNU"
            dic_informations["sourceType"] = None
            
            for i in range(len(dataframe_informations)):
                if "insee" in str(dataframe_informations.iloc[i, 0]).lower():
                    dic_informations["sourceName"] = "INSEE"
                    if not dataframe_informations.isna().iloc[i, 1]:
                        sourceInfo = str(dataframe_informations.iloc[i, 1]).split(",")
                        dic_informations["sourceType"] = sourceInfo[1] if len(sourceInfo) > 1 else None
                    break
                elif "drees" in str(dataframe_informations.iloc[i, 0]).lower():
                    dic_informations["sourceName"] = "DREES"
                    if not dataframe_informations.isna().iloc[i, 1]:
                        sourceInfo = str(dataframe_informations.iloc[i, 1]).split(",")
                        dic_informations["sourceType"] = sourceInfo[1] if len(sourceInfo) > 1 else None
                    break
        else:
            dic_informations["title"] = nom_fichier
            dic_informations["description"] = None
            dic_informations["sourceName"] = "INCONNU"
            dic_informations["sourceType"] = None            

    else:
        dic_headers = {}
        dic_informations["title"] = nom_fichier
        dic_informations["description"] = None
        dic_informations["sourceName"] = "INCONNU"
        dic_informations["sourceType"] = None

    low_gran_and_scope_tem = {nom_fichier: rechercherLowGranEtScope(liste_attributs_temporels, hierarchie_temps_spa)}
    low_gran_and_scope_spa = {nom_fichier: rechercherLowGranEtScope(liste_attributs_spatiaux, hierarchie_champs_spa)}

    creerDatasetUML(df,
                    nom_fichier,
                    extension,
                    low_gran_and_scope_spa[nom_fichier],
                    low_gran_and_scope_tem[nom_fichier],
                    liste_attributs_spatiaux,
                    liste_attributs_temporels,
                    liste_attributs_qualitatifs,
                    liste_attributs_quantitatifs,
                    headers, dic_headers, dic_informations)
    return


In [ ]:
# # dataset2 = "Opendata/Général/Education/pop-16ans-dipl6820_v2/pop-16ans-dipl6820_v2.xlsx"
# dataset = "Opendata/Général/Population/etatcivil2019_dec2019_csv/FD_DEC_2019.csv"

# fichier1 = dataset.split('/')[-1]
# nom_fichier, extension = fichier1.split('.')
# # fichier2 = dataset2.split('/')[-1]
# # nom_fichier2, extension2 = fichier2.split('.')

# try:
#     df = lf.find_type(dataset)[0]
#     if len(df.columns) < 5:
#         try:
#             df = pd.read_csv(dataset, sep=';', encoding='utf-8')
#         except:
#             df = pd.read_csv(dataset, sep=';', encoding='latin1')

#     process_dataframe(df,
#                         nom_fichier, 
#                         extension,
#                         hierarchie_temps_spa,
#                         hierarchie_champs_spa)

# except Exception as e:
#     print(f"Erreur lors du chargement du fichier {nom_fichier[:10]}: {e}")

In [ ]:
for dataset in datasets:
    fichier = dataset.split('/')[-1]
    nom_fichier, extension = fichier.split('.')
    print(f"Traitement du fichier {nom_fichier}")

    try:
        if extension == 'xlsx':
            listeSheets = pd.ExcelFile(dataset)
            nbSheets = len(listeSheets.sheet_names)
            for i in range(nbSheets):
                try:
                    df = lf.find_type(dataset, i)[0].ffill()
                    df, feuilleInvalides = chercherEntete(df)
                    if feuilleInvalides:
                        # print(f"\t- Feuille {i} invalide, passage à la suivante.")
                        continue

                    process_dataframe(df, 
                                      f"{nom_fichier}_sheet{i}", 
                                      extension,
                                      hierarchie_temps_spa, 
                                      hierarchie_champs_spa,
                                      dataset)

                except Exception as e:
                    print(f"\t  /!\\ Erreur lors de la lecture de la feuille {i} : {e}")
                    continue
                
        elif extension == 'csv':
            try:
                df = lf.find_type(dataset)[0]
                if len(df.columns) < 5:
                    try:
                        df = pd.read_csv(dataset, sep=';', encoding='utf-8')
                    except:
                        df = pd.read_csv(dataset, sep=';', encoding='latin1')

                process_dataframe(df,
                                  nom_fichier, 
                                  extension,
                                  hierarchie_temps_spa,
                                  hierarchie_champs_spa)

            except Exception as e:
                print(f"Erreur lors du chargement du fichier {nom_fichier[:10]}: {e}")
                continue
        else:
            print(f"Extension non supportée pour {nom_fichier}")
            continue
    except Exception as e:
        print(f"Erreur générale sur {nom_fichier}: {e}")
        continue


In [ ]:
del champs_ranges, dataset, datasets, df, dic_hierarchise, extension, feuilleInvalides, fichier, hierarchie_champs_spa, hierarchie_temps_spa,i, listeRegexTemporel, listeSheets, nbSheets, nom_fichier, regexAnnee, regexDate, regexHeure, regexJour, regexMois, regexTrim, temps_ranges

In [15]:
import pandas as pd

df = pd.DataFrame({
    'pays': ['france', 'france métropolitaine', 'france d\'outre-mer'],
    'regions': ['auvergne-rhône-alpes', 'bretagne', 'ile-de-france'],
    'departements': ['69', '35', '75'],
    'communes': ['lyon', '  rennes', 'paris'],
    'quartiers': ['quartier1', 'quartier2', 'quartier3'],
    'iris': ['iris1', 'iris2', 'iris3']})

print("Exemple de DataFrame créé pour la démonstration :")

for index, row in df.iterrows():
    print(f"{row.index}")

Exemple de DataFrame créé pour la démonstration :
Index(['pays', 'regions', 'departements', 'communes', 'quartiers', 'iris'], dtype='object')
Index(['pays', 'regions', 'departements', 'communes', 'quartiers', 'iris'], dtype='object')
Index(['pays', 'regions', 'departements', 'communes', 'quartiers', 'iris'], dtype='object')
